# Uber Eats Operational & Customer Intelligence System
## Exploratory Data Analysis (EDA)

**Purpose:** Explore the synthetic datasets, assess data quality, and understand customer, order, delivery, restaurant, and review patterns.

**Important:** These are synthetic educational data, not real Uber Eats operational records.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
print("Project directory:", PROJECT_ROOT)
print("Raw data directory exists:", DATA_DIR.exists())

In [ ]:
# Load available raw datasets
datasets = {}
for file in sorted(DATA_DIR.glob("*.csv")):
    name = file.stem
    datasets[name] = pd.read_csv(file)
    print(f"{name}: {datasets[name].shape[0]} rows, {datasets[name].shape[1]} columns")

if not datasets:
    raise FileNotFoundError(f"No CSV datasets found in {DATA_DIR}")

## 1. Dataset structure and data quality

Review column types, missing values, and duplicate rows for each source dataset.

In [ ]:
for name, df in datasets.items():
    print("\\n" + "=" * 65)
    print(f"DATASET: {name}")
    print("=" * 65)
    display(df.head())
    print("\\nColumn types:")
    print(df.dtypes)
    print("\\nMissing values:")
    print(df.isna().sum())
    print("\\nDuplicate rows:", df.duplicated().sum())

## 2. Numeric summaries

Summary statistics help identify typical values, spread, and possible unusual values. These summaries alone do not prove that any value is erroneous.

In [ ]:
for name, df in datasets.items():
    numeric = df.select_dtypes(include="number")
    if not numeric.empty:
        print(f"\\n{name} — numeric summary")
        display(numeric.describe().T.round(2))

## 3. Order amounts and order status

In [ ]:
orders = datasets.get("orders")
if orders is not None:
    display(orders.head())
    if "order_amount" in orders.columns:
        print(orders["order_amount"].describe())
        orders["order_amount"].dropna().plot(
            kind="hist", bins=20, edgecolor="black",
            title="Distribution of Order Amounts"
        )
        plt.xlabel("Order amount")
        plt.ylabel("Number of orders")
        plt.tight_layout()
        plt.show()
    if "order_status" in orders.columns:
        print("Order status counts:")
        display(orders["order_status"].value_counts(dropna=False).to_frame("count"))
else:
    print("orders.csv was not found.")

## 4. Delivery-time analysis

In [ ]:
deliveries = datasets.get("deliveries")
if deliveries is not None:
    display(deliveries.head())
    if "delivery_time_min" in deliveries.columns:
        print(deliveries["delivery_time_min"].describe())
        deliveries["delivery_time_min"].dropna().plot(
            kind="hist", bins=20, edgecolor="black",
            title="Distribution of Delivery Times"
        )
        plt.xlabel("Delivery time (minutes)")
        plt.ylabel("Number of deliveries")
        plt.tight_layout()
        plt.show()
    if {"distance_km", "delivery_time_min"}.issubset(deliveries.columns):
        deliveries.plot(
            kind="scatter", x="distance_km", y="delivery_time_min",
            title="Delivery Time vs Distance"
        )
        plt.tight_layout()
        plt.show()
else:
    print("deliveries.csv was not found.")

## 5. Review ratings

In [ ]:
reviews = datasets.get("reviews")
if reviews is not None:
    display(reviews.head())
    if "review_rating" in reviews.columns:
        print("Rating counts:")
        display(reviews["review_rating"].value_counts(dropna=False).sort_index().to_frame("count"))
        reviews["review_rating"].dropna().plot(
            kind="hist", bins=5, edgecolor="black",
            title="Distribution of Review Ratings"
        )
        plt.xlabel("Review rating")
        plt.ylabel("Number of reviews")
        plt.tight_layout()
        plt.show()
    if "review_text" in reviews.columns:
        print("Missing review text:", reviews["review_text"].isna().sum())
        print("Unique review texts:", reviews["review_text"].nunique(dropna=True))
else:
    print("reviews.csv was not found.")

## 6. Correlation analysis

Correlations describe linear associations between numeric columns; they do not establish causation.

In [ ]:
for name in ["orders", "deliveries", "restaurants", "payments"]:
    df = datasets.get(name)
    if df is None:
        continue
    numeric = df.select_dtypes(include="number")
    if numeric.shape[1] >= 2:
        print(f"\\n{name} — numeric correlations")
        display(numeric.corr().round(3))
        corr = numeric.corr()
        fig, ax = plt.subplots(figsize=(max(5, len(corr.columns)), max(4, len(corr.columns) * 0.7)))
        image = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
        ax.set_xticks(range(len(corr.columns)))
        ax.set_yticks(range(len(corr.columns)))
        ax.set_xticklabels(corr.columns, rotation=45, ha="right")
        ax.set_yticklabels(corr.columns)
        fig.colorbar(image, ax=ax, label="Correlation")
        ax.set_title(f"{name.title()} Numeric Correlations")
        fig.tight_layout()
        plt.show()

## 7. Key findings and limitations

Use the executed outputs above to record findings. Initial known project limitations include:

- The data is synthetic and should not be treated as representative of real platform operations.
- Order timestamps cover only a very short time window, so meaningful hourly or seasonal forecasting is not possible.
- Review texts are repetitive and can have conflicting rating labels, limiting sentiment-model reliability.
- Traffic, weather, preparation time, and pickup waiting time are unavailable for delivery-time modeling.
- Correlation and model feature importance do not prove causal relationships.

### Conclusion

This notebook provides an exploratory overview of the available raw data. It complements the project’s existing feature-engineering, clustering, sentiment, regression, and forecasting scripts. Add concise, evidence-based findings after running all cells.